# Morocco WC2022: Defensive Efficiency & Attacking Precision
### Quantifying the Blueprint: How Morocco Reached the 2022 World Cup Semifinals
*Hachad Solutions LLC · March 2026 · StatsBomb Open Data*

## Motivation

Morocco became the first African and Arab nation to reach a World Cup  semifinal in 2022, eliminating Belgium, Spain, and Portugal along the way before falling to France. Post-tournament tactical analysis, the FIFA Technical Study Group report, and academic profiling work all highlighted a compact defensive structure, disciplined low-to-mid block, and effective transitions as the hallmarks of their run.

This project takes those observations as a starting point and interrogates them with StatsBomb event and 360 data  to see what the numbers actually say at the event level. All existing analysis used aggregated post-match report data. This project uses StatsBomb's event-level and 360 spatial data — a level of granularity no prior published analysis has applied to this specific question.

See `REFERENCES.md` for the prior work that informed this framing.

## Analytical Questions

This project asks three open questions; the answers will emerge from the data:

**Q1: Defensive structure:**  
What does Morocco's defensive behaviour actually look like at the event level across 7 matches? Where did they win the ball back, and what quality of chance did opponents generate as a result? Which individual players or spatial zones drove that structure?

**Q2: Attacking efficiency:**  
Was Morocco's attacking output a reflection of genuine chance quality, or did they outperform their xG? What do their goals look like spatially compared to their baseline expected threat?

**Q3: Broader applicability and WC2026 relevance:**
Can Morocco's WC2022 performance be characterised as a replicable blueprint rather than a one-off? Using Morocco's quantified defensive and offensive profile as a reference point, which qualified WC2026 nations share a similar competitive tier and could plausibly adopt a similar approach? This phase draws on additional data sources beyond StatsBomb to profile WC2026 participants and map them against Morocco's blueprint.

## Notebook Structure

1. Setup & Data Loading
2. Morocco Matches Overview
3. Defensive Analysis
   - 3a. Pressure & defensive intervention map
   - 3b. Opponent shot quality: location and xG
   - 3c. Individual contributions to defensive structure
4. Offensive Analysis
   - 4a. Morocco shot map and xG
   - 4b. Goals vs. xG baseline: finishing efficiency
5. Tournament-level Comparison (all 32 teams)
6. Findings & Conclusions

## Data Source

**StatsBomb Open Data: FIFA World Cup 2022**  
`competition_id = 43` · `season_id = 106`  
64 matches · 32 teams · Full event + 360 freeze-frame data  
https://github.com/statsbomb/open-data

*Data accessed via `statsbombpy` SDK. Per StatsBomb's user agreement, 
this project credits StatsBomb as the data source for all analysis 
derived from their data.*

---
## 1. Setup & Data Loading

In [1]:
# Data source
from statsbombpy import sb

# Data wrangling
import pandas as pd
import numpy as np

# Database
import sqlite3
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from mplsoccer import Pitch, VerticalPitch

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
%matplotlib inline

### 1.1 Load Morocco Matches

In [4]:
# Load all WC2022 matches
matches = sb.matches(competition_id=43, season_id=106)

# Filter to Morocco matches only
morocco_matches = matches[
    (matches['home_team'] == 'Morocco') |
    (matches['away_team'] == 'Morocco')    
].reset_index(drop=True)

# Display Match Overview
morocco_matches[[
    'match_id', 'match_date', 'competition_stage', 'home_team', 'home_score', 'away_team', 'away_score'
]].sort_values('match_date')

/Users/tarekhachad/dev/morocco-wc22-analysis/venv/lib/python3.11/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,competition_stage,home_team,home_score,away_team,away_score
5,3857277,2022-11-23,Group Stage,Morocco,0,Croatia,0
4,3857283,2022-11-27,Group Stage,Belgium,0,Morocco,2
6,3857276,2022-12-01,Group Stage,Canada,1,Morocco,2
3,3869220,2022-12-06,Round of 16,Morocco,0,Spain,0
0,3869486,2022-12-10,Quarter-finals,Morocco,1,Portugal,0
2,3869552,2022-12-14,Semi-finals,France,2,Morocco,0
1,3869684,2022-12-17,3rd Place Final,Croatia,2,Morocco,1
